# Tutorial 04 — Brain MRI Contrasts

**What you will learn:**
- Why MRI looks completely different from CT
- What T1, T2, FLAIR, and SWI actually mean
- How the same brain can look dramatically different in each contrast
- What skull-stripping is and why it matters for brain AI models

**No GPU needed.** We use the eight pre-generated brain MRI volumes in `outputs/maisi2_mr_brain_all_contrasts/`.

---
> **Safety reminder:** All images are fully synthetic — generated by AI, not from any real patient.

## MRI vs CT — a fundamental difference

| Property | CT | MRI |
|---|---|---|
| Physics | X-ray absorption | Radiofrequency pulses + magnetic field |
| Values | Hounsfield Units — physically fixed | Arbitrary signal intensity — depends on the scan settings |
| Bone | Bright white (high HU) | Dark (bone has little water) |
| Brain tissue | Difficult to distinguish grey/white matter | Excellent soft-tissue contrast |
| Radiation | Ionising (X-ray) | Non-ionising |
| Scan time | Seconds | Minutes |

**The key difference for AI:** CT values are absolute (you can compare voxel values between patients). MRI values are relative — the same tissue will have a different signal in every scanner, every protocol, every centre. This is why MRI harmonisation is a major research challenge.

### What is a "contrast" in MRI?

An MRI scanner can use different **pulse sequences** (timing patterns of radiofrequency pulses). Each sequence emphasises different tissue properties:

- **T1** — emphasises longitudinal relaxation. Fat/white matter are bright. CSF is dark.
- **T2** — emphasises transverse relaxation. Water/CSF is very bright. White matter is dark.
- **FLAIR** — like T2 but with CSF suppressed (made dark). Lesions near CSF become visible.
- **SWI** — sensitive to magnetic susceptibility effects. Blood products, iron, and veins are dark.

In [ ]:
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PROJECT_ROOT = Path().resolve()
for _ in range(6):
    if (PROJECT_ROOT / 'outputs').exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

BRAIN_DIR = PROJECT_ROOT / 'outputs/maisi2_mr_brain_all_contrasts'

%matplotlib inline
plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
                     'text.color': 'white', 'axes.labelcolor': '#aaa',
                     'xtick.color': '#aaa', 'ytick.color': '#aaa'})

def norm_mri(data):
    """Percentile normalization: maps foreground tissue to [0,1]."""
    fg = data[data > data.mean() * 0.1]
    if fg.size == 0:
        return np.clip(data / (data.max() + 1e-6), 0, 1)
    lo, hi = np.percentile(fg, 1), np.percentile(fg, 99)
    return np.clip((data - lo) / (hi - lo + 1e-6), 0, 1)

print('Setup complete. Brain MRI directory:', BRAIN_DIR.relative_to(PROJECT_ROOT))

In [ ]:
# ── Find all available brain MRI files ────────────────────────────────────

contrast_patterns = [
    ('mri_t1',                 'T1 (whole brain)'),
    ('mri_t2',                 'T2 (whole brain)'),
    ('mri_flair',              'FLAIR (whole brain)'),
    ('mri_swi',                'SWI (whole brain)'),
    ('mri_t1_skull_stripped',  'T1 skull-stripped'),
    ('mri_t2_skull_stripped',  'T2 skull-stripped'),
    ('mri_flair_skull_stripped','FLAIR skull-stripped'),
    ('mri_swi_skull_stripped', 'SWI skull-stripped'),
]

files = {}
for pattern, label in contrast_patterns:
    # Match filename by pattern (exclude skull_stripped when looking for bare contrast)
    matches = sorted(BRAIN_DIR.glob(f'*{pattern}_seed*.nii.gz'))
    # For bare contrasts, exclude skull_stripped versions
    if 'skull_stripped' not in pattern:
        matches = [m for m in matches if 'skull_stripped' not in m.name]
    if matches:
        files[pattern] = (matches[0], label)
        print(f'  {label:<28} → {matches[0].name}')
    else:
        print(f'  {label:<28} → NOT FOUND')

print(f'\nFound {len(files)} contrast files.')

In [ ]:
# ── First look at T1 in three planes ──────────────────────────────────────

t1_path, t1_label = files['mri_t1']
t1 = nib.load(str(t1_path)).get_fdata(dtype=np.float32)
cx, cy, cz = [s // 2 for s in t1.shape]

print(f'T1 shape : {t1.shape}  spacing : {nib.load(str(t1_path)).header.get_zooms()[:3]}')
print(f'Intensity range: [{t1.min():.0f}, {t1.max():.1f}]')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
planes = [
    ('Sagittal', np.rot90(t1[cx, :, :])),
    ('Coronal',  np.rot90(t1[:, cy, :])),
    ('Axial',    np.rot90(t1[:, :, cz])),
]
for ax, (plane, slc) in zip(axes, planes):
    ax.imshow(norm_mri(slc), cmap='gray', vmin=0, vmax=1, aspect='equal')
    ax.set_title(plane, fontsize=12)
    ax.axis('off')

fig.suptitle('T1-weighted brain MRI — three anatomical planes', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut04_t1_three_planes.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()

### Reading a T1 brain scan

In T1-weighted MRI, the brightness tells you about tissue **fat content and myelin**:

- **Bright white** = white matter (heavily myelinated axons), fat, enhancing lesions
- **Medium grey** = grey matter (cortex, basal ganglia, cerebellum)
- **Dark** = cerebrospinal fluid (CSF) in the ventricles and sulci
- **Black** = air (sinuses), cortical bone (skull)

Look for the **ventricles** (dark butterfly-shaped structure in the axial view) — those are the fluid-filled spaces inside the brain.

In [ ]:
# ── T1 / T2 / FLAIR / SWI — same slice, four contrasts ───────────────────
#
# This is the most educational comparison you can do with brain MRI.

main_contrasts = ['mri_t1', 'mri_t2', 'mri_flair', 'mri_swi']
loaded = {k: nib.load(str(files[k][0])).get_fdata(dtype=np.float32)
          for k in main_contrasts if k in files}

# Use the axial centre slice — good for all four contrasts
ref = loaded['mri_t1']
z = ref.shape[2] // 2

fig, axes = plt.subplots(1, len(loaded), figsize=(5 * len(loaded), 5.5))
axes = axes if len(loaded) > 1 else [axes]

descriptions = {
    'mri_t1':    'T1\nWhite matter = bright\nCSF = dark',
    'mri_t2':    'T2\nCSF = very bright\nWhite matter = grey',
    'mri_flair': 'FLAIR\nCSF suppressed (dark)\nLesions near CSF visible',
    'mri_swi':   'SWI\nBlood/iron = dark\nVeins visible',
}

for ax, (k, vol) in zip(axes, loaded.items()):
    slc = vol[:, :, z]
    ax.imshow(np.rot90(norm_mri(slc)), cmap='gray', vmin=0, vmax=1, aspect='equal')
    ax.set_title(descriptions.get(k, k), fontsize=10)
    ax.axis('off')

fig.suptitle('Same brain, same slice — four MRI contrast types', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut04_four_contrasts.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Notice: CSF (ventricles) changes from dark (T1) → bright (T2) → dark again (FLAIR).')

In [ ]:
# ── Difference maps — what actually changes between contrasts ─────────────
#
# Subtracting two normalised contrasts shows WHAT changed, not just HOW it looks.

if 'mri_t1' in loaded and 'mri_t2' in loaded:
    t1_norm = norm_mri(loaded['mri_t1'][:, :, z])
    t2_norm = norm_mri(loaded['mri_t2'][:, :, z])
    diff_t1_t2 = t1_norm - t2_norm   # positive where T1 > T2 (white matter, fat)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(np.rot90(t1_norm), cmap='gray', vmin=0, vmax=1, aspect='equal')
    axes[0].set_title('T1', fontsize=12)
    axes[0].axis('off')

    axes[1].imshow(np.rot90(t2_norm), cmap='gray', vmin=0, vmax=1, aspect='equal')
    axes[1].set_title('T2', fontsize=12)
    axes[1].axis('off')

    axes[2].imshow(np.rot90(diff_t1_t2), cmap='RdBu_r', vmin=-0.6, vmax=0.6, aspect='equal')
    axes[2].set_title('T1 − T2\nBlue=T2 brighter (CSF)  Red=T1 brighter (white matter)', fontsize=9)
    axes[2].axis('off')

    fig.suptitle('T1 vs T2 — and their difference', fontsize=13)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'figures/tut04_t1_t2_diff.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('Blue regions = where T2 is brighter than T1 (mostly CSF).\nRed regions = where T1 is brighter (mostly white matter).')

In [ ]:
# ── Skull-stripped vs whole-brain T1 ──────────────────────────────────────
#
# Skull-stripping removes the skull, scalp, and non-brain tissue,
# leaving only brain parenchyma. This is standard preprocessing for
# almost every brain AI model.

if 'mri_t1' in files and 'mri_t1_skull_stripped' in files:
    t1_whole  = nib.load(str(files['mri_t1'][0])).get_fdata(dtype=np.float32)
    t1_strip  = nib.load(str(files['mri_t1_skull_stripped'][0])).get_fdata(dtype=np.float32)
    z = t1_whole.shape[2] // 2

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    axes[0].imshow(np.rot90(norm_mri(t1_whole[:, :, z])), cmap='gray', vmin=0, vmax=1, aspect='equal')
    axes[0].set_title('T1 — whole brain (with skull)', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(np.rot90(norm_mri(t1_strip[:, :, z])), cmap='gray', vmin=0, vmax=1, aspect='equal')
    axes[1].set_title('T1 skull-stripped\n(brain tissue only)', fontsize=11)
    axes[1].axis('off')

    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'figures/tut04_skull_strip.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('The stripped version has all non-brain signal set to zero.')
else:
    print('Skull-stripped file not found — run brain_all preset to generate all 8 contrasts.')

### Why skull-strip?

Brain AI models that do segmentation, registration, or lesion detection are usually trained only on the brain parenchyma. Including the skull:

1. Adds irrelevant signal that can bias training
2. Causes skull-related artefacts during registration
3. Increases image size unnecessarily

Tools like **FreeSurfer**, **FSL BET**, and **HD-BET** are commonly used to strip skulls from real MRI. Having skull-stripped synthetic data means you can train a model that expects stripped input without needing to pre-process real data first.

In [ ]:
# ── All 8 available contrasts in one figure ────────────────────────────────

available = {k: (path, label) for k, (path, label) in files.items()}
n = len(available)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes_flat = np.array(axes).ravel()

for ax, (pattern, (path, label)) in zip(axes_flat, available.items()):
    vol = nib.load(str(path)).get_fdata(dtype=np.float32)
    z = vol.shape[2] // 2
    ax.imshow(np.rot90(norm_mri(vol[:, :, z])), cmap='gray', vmin=0, vmax=1, aspect='equal')
    ax.set_title(label, fontsize=9)
    ax.axis('off')

for ax in axes_flat[n:]:
    ax.axis('off')

fig.suptitle('All available brain MRI contrasts — same brain, same seed, different physics', fontsize=12)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut04_all_contrasts.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'{n} contrasts shown.')

In [ ]:
# ── Intensity profiles through the brain centre ───────────────────────────
#
# A line profile shows how signal changes as you cross the brain.
# This is a direct measurement of what changes between contrasts.

main_keys = [k for k in ['mri_t1','mri_t2','mri_flair','mri_swi'] if k in loaded]
colors_line = ['#4dabf7','#ff4d4d','#69db7c','#ffd43b']

fig, ax = plt.subplots(figsize=(13, 4))

for k, color in zip(main_keys, colors_line):
    vol = loaded[k]
    z = vol.shape[2] // 2
    y = vol.shape[1] // 2
    profile = norm_mri(vol[:, y, z])   # line through centre of the axial slice
    ax.plot(profile, color=color, linewidth=1.5, label=k.replace('mri_', '').upper(), alpha=0.9)

ax.set_xlabel('Voxel position (left → right through brain)')
ax.set_ylabel('Normalised signal intensity')
ax.set_title('Signal profile across the brain centre — contrasts diverge at CSF (skull/ventricles)')
ax.legend(fontsize=10)
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures/tut04_intensity_profiles.png', dpi=130, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('The biggest divergence occurs at the skull and ventricles.')

## Summary

| Contrast | CSF | White matter | Key use |
|---|---|---|---|
| T1 | Dark | Bright | Anatomy, post-contrast enhancement |
| T2 | Bright | Grey | Oedema, lesions, fluid |
| FLAIR | Dark (suppressed) | Grey | White matter lesions near CSF |
| SWI | Variable (dark veins) | Intermediate | Micro-bleeds, venous anatomy |

### The challenge this creates for AI

If you train a brain tumour model on T1 images only, it will likely fail on T2 images because the tissue intensities are reversed in some regions. Multi-contrast models (trained on T1+T2+FLAIR together) are more robust — and synthetic multi-contrast data from MAISI helps train them without needing large multi-contrast real datasets.

## What's next?

**Tutorial 05** — Run your own generation: use `MAISI_RT_Generate.py`, explore seeds, and understand what happens inside the YAML config.